# Flood Vulnerability Classification Model

## Objective
Train a machine learning model to predict long-term flood-prone zones near rivers for sustainable urban planning.

**Target Variable:** `Flood_Prone` (0 = Not flood prone, 1 = Flood prone)

**Model:** RandomForestClassifier with 80-20 train-test split

**Expected Output:** Trained model saved to `ml/flood_model.pkl`

## 1. Import Required Libraries

In [ ]:
# Data processing and analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
import joblib

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✓ All libraries imported successfully')

## 2. Load Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('ml/dataset.csv')

# Display basic information
print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
print(df.head())

print('\nColumn Names and Types:')
print(df.dtypes)

print('\nDataset Info:')
print(df.info())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Check for missing values
print('Missing Values:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found')

print('\nMissing Values Percentage:')
missing_pct = (df.isnull().sum() / len(df) * 100)
print(missing_pct[missing_pct > 0] if missing_pct.sum() > 0 else 'No missing values')

In [ ]:
# Display summary statistics
print('Summary Statistics:')
print(df.describe())

In [ ]:
# Analyze class distribution of target variable
print('Target Variable Distribution (Flood_Prone):')
print(df['Flood_Prone'].value_counts())

print('\nClass Distribution (%):')
print(df['Flood_Prone'].value_counts(normalize=True) * 100)

# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
df['Flood_Prone'].value_counts().plot(kind='bar', ax=axes[0], color=['#22C55E', '#EF4444'])
axes[0].set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Flood Prone', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_xticklabels(['Not Prone (0)', 'Prone (1)'], rotation=0)

# Pie chart
df['Flood_Prone'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                       colors=['#22C55E', '#EF4444'], labels=['Not Prone', 'Prone'])
axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print('✓ Class distribution analyzed')

## 4. Data Preprocessing

In [ ]:
# Make a copy for preprocessing
df_clean = df.copy()

# Handle missing values
print('Before preprocessing:')
print('Shape:', df_clean.shape)
print('Missing values:', df_clean.isnull().sum().sum())

# Drop rows with missing values in critical columns (adjust as needed)
# Or fill with appropriate values
# Option 1: Drop rows with any missing values
df_clean = df_clean.dropna()

print('\nAfter preprocessing:')
print('Shape:', df_clean.shape)
print('Rows removed:', df.shape[0] - df_clean.shape[0])

print('✓ Missing values handled')

In [ ]:
# Separate features (X) and target (y)
# Assuming 'Flood_Prone' is the target column
y = df_clean['Flood_Prone']
X = df_clean.drop('Flood_Prone', axis=1)

print('Feature variables (X):')
print('Shape:', X.shape)
print('\nColumns:')
print(X.columns.tolist())

print('\nTarget variable (y):')
print('Shape:', y.shape)
print('Unique values:', y.unique())

print('\n✓ Features and target separated')

In [ ]:
# Check feature datatypes and ensure numeric types
print('Feature Data Types:')
print(X.dtypes)

# Convert categorical columns to numeric if needed
# Example: if there are object dtypes, we encode them
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

if categorical_cols:
    print(f'\nCategorical columns found: {categorical_cols}')
    # One-hot encode categorical variables
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    print(f'After encoding, X shape: {X.shape}')
else:
    print('\nNo categorical columns. All features are numeric.')

print('\n✓ Feature encoding complete')

## 5. Train-Test Split

In [ ]:
# Split dataset: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Preserve class distribution
)

print('Train-Test Split Results:')
print(f'Training set: {X_train.shape}')
print(f'Testing set: {X_test.shape}')

print('\nTraining set class distribution:')
print(y_train.value_counts())

print('\nTesting set class distribution:')
print(y_test.value_counts())

print('\n✓ Train-test split complete (80-20)')

## 6. Model Training

In [ ]:
# Initialize RandomForestClassifier
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,  # Use all available processors
    verbose=1
)

print('Model Configuration:')
print(model.get_params())

In [ ]:
# Train the model
print('Training model...')
model.fit(X_train, y_train)
print('✓ Model training complete')

## 7. Model Evaluation

In [ ]:
# Make predictions on training and testing sets
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print('Predictions generated for both train and test sets')
print(f'Training predictions shape: {y_train_pred.shape}')
print(f'Testing predictions shape: {y_test_pred.shape}')

In [ ]:
# Calculate accuracy scores
train_accuracy = metrics.accuracy_score(y_train, y_train_pred)
test_accuracy = metrics.accuracy_score(y_test, y_test_pred)

print('Accuracy Scores:')
print(f'Training Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)')
print(f'Testing Accuracy:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

if train_accuracy - test_accuracy > 0.1:
    print('\n⚠️  Model may be overfitting (large gap between train and test accuracy)')
else:
    print('\n✓ Model generalization looks good')

In [ ]:
# Detailed evaluation metrics on test set
print('\n' + '='*60)
print('DETAILED EVALUATION METRICS (Test Set)')
print('='*60)

precision = metrics.precision_score(y_test, y_test_pred)
recall = metrics.recall_score(y_test, y_test_pred)
f1 = metrics.f1_score(y_test, y_test_pred)
roc_auc = metrics.roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

print(f'\nAccuracy:  {test_accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {f1:.4f}')
print(f'ROC-AUC:   {roc_auc:.4f}')

print('\n' + '='*60)
print('CLASSIFICATION REPORT')
print('='*60)
print(metrics.classification_report(y_test, y_test_pred, target_names=['Not Prone', 'Prone']))

In [ ]:
# Confusion Matrix
cm = metrics.confusion_matrix(y_test, y_test_pred)

print('\nConfusion Matrix:')
print(cm)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Not Prone', 'Prone'],
            yticklabels=['Not Prone', 'Prone'])
plt.title('Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

# Calculate True Positives, False Negatives, etc.
tn, fp, fn, tp = cm.ravel()
print(f'\nTrue Negatives (TN):  {tn}')
print(f'False Positives (FP): {fp}')
print(f'False Negatives (FN): {fn}')
print(f'True Positives (TP):  {tp}')

## 8. Feature Importance Analysis

In [ ]:
# Extract feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print('Top 15 Most Important Features:')
print(feature_importance.head(15))

# Plot feature importance
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['Importance'], color='#3B82F6')
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Top 15 Feature Importances', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f'\n✓ Feature importance calculated for {len(feature_importance)} features')

## 9. Save Trained Model

In [ ]:
import os

# Create ml directory if it doesn't exist
os.makedirs('ml', exist_ok=True)

# Save the trained model
model_path = 'ml/flood_model.pkl'
joblib.dump(model, model_path)

print(f'✓ Model saved to: {model_path}')
print(f'  File size: {os.path.getsize(model_path) / 1024:.2f} KB')

# Verify the model can be loaded
loaded_model = joblib.load(model_path)
print(f'\n✓ Model successfully loaded and verified')
print(f'  Loaded model type: {type(loaded_model)}')

In [ ]:
# Save model metadata and feature names for future predictions
model_metadata = {
    'model_type': 'RandomForestClassifier',
    'n_estimators': 100,
    'test_accuracy': float(test_accuracy),
    'test_precision': float(precision),
    'test_recall': float(recall),
    'test_f1': float(f1),
    'test_roc_auc': float(roc_auc),
    'feature_names': X.columns.tolist(),
    'n_features': X.shape[1],
    'training_samples': X_train.shape[0],
    'testing_samples': X_test.shape[0]
}

# Save metadata
import json
metadata_path = 'ml/model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(model_metadata, f, indent=2)

print(f'✓ Model metadata saved to: {metadata_path}')
print('\nMetadata:')
for key, value in model_metadata.items():
    if not isinstance(value, list):
        print(f'  {key}: {value}')

## 10. Model Summary & Next Steps

### 🎯 Model Objective
This RandomForest model predicts long-term flood susceptibility based on terrain and hydrological indicators for sustainable urban planning.

### 📊 Performance Summary
- **Test Accuracy:** ≈ {test_accuracy*100:.1f}%
- **Precision:** ≈ {precision:.3f}
- **Recall:** ≈ {recall:.3f}
- **F1-Score:** ≈ {f1:.3f}
- **ROC-AUC:** ≈ {roc_auc:.3f}

### 📁 Deliverables
1. **Trained Model:** `ml/flood_model.pkl`
2. **Model Metadata:** `ml/model_metadata.json`
3. **Feature Importances:** Analyzed and visualized

### 🚀 Next Steps for Production
1. **Integration:** Load `flood_model.pkl` in your React/Cesium dashboard to make real-time predictions
2. **API Endpoint:** Create a Python API (Flask/FastAPI) to serve predictions
3. **Model Monitoring:** Track model performance on new data
4. **Hyperparameter Tuning:** Use GridSearchCV to optimize model parameters
5. **Feature Engineering:** Explore additional terrain and hydrological features

### 💡 Key Insights
- The top features identify the most critical indicators for flood susceptibility
- Balanced precision and recall indicate good model generalization
- Feature importance guides future data collection and urban planning decisions

---
**Model Created:** RiverTwin AI Flood Vulnerability Prediction System

**Use Case:** Urban Planning & Disaster Risk Reduction